In [ ]:
import torch
import numpy as np
import lightning as L
from pathlib import Path
from collections import defaultdict
from omegaconf import OmegaConf
from hydra.utils import instantiate
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
import os
import sys

# --- CONFIGURATION ---
PROJECT_PATH = Path.cwd().parent.resolve()
BASE_PATHS = [
    Path(PROJECT_PATH / "logs/train/multiruns/2025-12-08_17-14-20"),
    Path(PROJECT_PATH / "logs/train/multiruns/2025-12-15_16-24-59")
]

if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

os.environ["PROJECT_ROOT"] = str(PROJECT_PATH)
CIFAR_C_DIR = Path(PROJECT_PATH / "data/CIFAR-10-C")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CORRUPTIONS = [
    "gaussian_noise", "shot_noise", "impulse_noise", "defocus_blur", "glass_blur",
    "motion_blur", "zoom_blur", "snow", "frost", "fog", "brightness", "contrast",
    "elastic_transform", "pixelate", "jpeg_compression"
]

# Standard CIFAR-10 Normalization
CIFAR_TRANSFORM = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))

def load_model_from_run(run_dir):
    config_path = run_dir / "csv" / "version_0" / "hparams.yaml"
    if not config_path.exists():
        return None, None
    cfg = OmegaConf.load(config_path)
    cfg.data.data_dir = str(PROJECT_PATH / "data")
    model = instantiate(cfg.model)
    ckpt_path = list(run_dir.glob("**/*.ckpt"))[0]
    checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["state_dict"])
    return model.to(DEVICE).eval(), cfg

def get_cifar_c_loader(corruption, severity, batch_size=512):
    """Directly loads .npy and returns a DataLoader."""
    images = np.load(CIFAR_C_DIR / f"{corruption}.npy")
    labels = np.load(CIFAR_C_DIR / "labels.npy")
    
    # Slice the 10,000 images for the specific severity
    start, end = (severity - 1) * 10000, severity * 10000
    images_sev = images[start:end]
    labels_sev = labels[start:end]
    
    # Convert to Tensor, permute to (B, C, H, W) and scale to [0, 1]
    images_tensor = torch.from_numpy(images_sev).permute(0, 3, 1, 2).float() / 255.0
    labels_tensor = torch.from_numpy(labels_sev).long()
    
    # Apply normalization
    images_tensor = CIFAR_TRANSFORM(images_tensor)
    
    dataset = TensorDataset(images_tensor, labels_tensor)
    return DataLoader(dataset, batch_size=batch_size, num_workers=8, pin_memory=True)

# Storage
grouped_results = defaultdict(lambda: defaultdict(list))
trainer = L.Trainer(accelerator="gpu", devices=1, logger=False)

# Pre-load labels if necessary, though get_cifar_c_loader handles it
for base_path in BASE_PATHS:
    print(f"Processing Base Path: {base_path}")
    for run_dir in filter(Path.is_dir, base_path.iterdir()):
        model, cfg = load_model_from_run(run_dir)
        if model is None: continue

        n_subset = cfg.data.get("train_subset", "full")
        dr_rate = cfg.model.get("dropout_rate", 0.0)
        group_key = f"subset_{n_subset}_dr_{dr_rate}"

        for corr in tqdm(CORRUPTIONS, desc=f"Run {run_dir.name}", leave=False):
            all_sev_probs = []
            
            for sev in range(1, 6):
                # Using the direct loader instead of the DataModule
                loader = get_cifar_c_loader(corr, sev)
                
                # trainer.predict accepts a dataloader directly
                batch_results = trainer.predict(model, dataloaders=loader)
                
                logits_list = [torch.from_numpy(b["logits"]) for b in batch_results]
                sev_logits = torch.cat(logits_list, dim=0)
                
                # BDL Averaging: mean(softmax(logits))
                sev_probs_mcd = torch.softmax(sev_logits, dim=-1)
                sev_probs_mean = sev_probs_mcd.mean(dim=1)
                all_sev_probs.append(sev_probs_mean)
            
            avg_probs = torch.stack(all_sev_probs).mean(dim=0)
            grouped_results[group_key][corr].append(avg_probs)
            
        del model
        torch.cuda.empty_cache()

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


Processing Base Path: /home/rossom/hydra-experiments/logs/train/multiruns/2025-12-08_17-14-20


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL omegaconf.base.ContainerMetadata was not an allowed global by default. Please use `torch.serialization.add_safe_globals([omegaconf.base.ContainerMetadata])` or the `torch.serialization.safe_globals([omegaconf.base.ContainerMetadata])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load 